In [1]:
from datasets import load_dataset
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt

# 1. Load dataset

ds = load_dataset("gamino/wiki_medical_terms")

/Users/Kitu/anaconda3/envs/d266myassignment/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 6861/6861 [00:00<00:00, 28308.42 examples/s]


In [2]:
ds.column_names

{'train': ['page_title', 'page_text', '__index_level_0__']}

In [ ]:


# Convert to Pandas for exploration (train split only as example)
df = ds["train"].to_pandas()
df

,page_title,page_text,__index_level_0__
0,Paracetamol poisoning,"Paracetamol poisoning, also known as acetamino...",0
1,Acromegaly,Acromegaly is a disorder that results from exc...,1
2,Actinic keratosis,"Actinic keratosis (AK), sometimes called solar...",2
3,Congenital adrenal hyperplasia,Congenital adrenal hyperplasia (CAH) is a grou...,3
4,Adrenocortical carcinoma,Adrenocortical carcinoma (ACC) is an aggressi...,4
...,...,...,...
6856,Gephyrophobia,Gephyrophobia is the anxiety disorder or speci...,7271
6857,Coronary artery bypass surgery,"Coronary artery bypass surgery, also known as ...",7272
6858,Unemployment,"Unemployment, according to the OECD (Organisat...",7273
6859,Surgical instrument,A surgical instrument is a tool or device for ...,7274


In [8]:
df['page_title'].unique()

array(['Paracetamol poisoning', 'Acromegaly', 'Actinic keratosis', ...,
       'Unemployment', 'Surgical instrument', 'Occipital neuralgia'],
      dtype=object)

In [12]:
df["page_title"].value_counts()


page_title
Fasciculation                   4
Precocious puberty              3
Hypokinesia                     3
Benign prostatic hyperplasia    3
Barretts esophagus              3
                               ..
Nasal concha                    1
Uterine hypoplasia              1
Urolagnia                       1
Restrictive lung disease        1
Occipital neuralgia             1
Name: count, Length: 6762, dtype: int64

In [14]:
print("Does not have good distribution of data and categories are sprase ")

Does not have good distribution of data and categories are sprase 


In [15]:
print("This is a great datset for GanBERT")
print("We could try running clustering technique to categories the label?? ")


This is a great datset for GanBERT
We could try running clustering technique to categories the label?? 


In [22]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset("gamino/wiki_medical_terms", split="train")
print(dataset[0])
# {'title': 'Acromegaly', 'text': 'Acromegaly is a disorder that ...'}

{'page_title': 'Paracetamol poisoning', 'page_text': 'Paracetamol poisoning, also known as acetaminophen poisoning, is caused by excessive use of the medication paracetamol (acetaminophen). Most people have few or non-specific symptoms in the first 24 hours following overdose. These include feeling tired, abdominal pain, or nausea. This is typically followed by a couple of days without any symptoms, after which yellowish skin, blood clotting problems, and confusion occurs as a result of liver failure. Additional complications may include kidney failure, pancreatitis, low blood sugar, and lactic acidosis. If death does not occur, people tend to recover fully over a couple of weeks. Without treatment, death from toxicity occurs 4 to 18 days later.Paracetamol poisoning can occur accidentally or as an attempt to die by suicide. Risk factors for toxicity include alcoholism, malnutrition, and the taking of certain other hepatotoxic medications. Liver damage results not from paracetamol itsel

Create embeddings

You need embeddings that capture semantic meaning of medical text. Options:

Hugging Face models: e.g. sentence-transformers/all-MiniLM-L6-v2 (fast, general purpose).

Domain-specific embeddings: e.g. BioClinicalBERT (emilyalsentzer/Bio_ClinicalBERT) or PubMedBERT for better medical performance.

In [ ]:
from sentence_transformers import SentenceTransformer
from huggingface_hub import hf_hub_download

# Patch for legacy code expecting cached_download
if not hasattr(huggingface_hub, "cached_download"):
    huggingface_hub.cached_download = hf_hub_download
# pick model (general first, switch to medical later if needed)
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Use the title as short description (could concatenate with text[:512])
corpus = [row["title"] for row in dataset]
embeddings = embedder.encode(corpus, show_progress_bar=True)

ImportError: cannot import name 'cached_download' from 'huggingface_hub' (/Users/Kitu/anaconda3/envs/d266myassignment/lib/python3.9/site-packages/huggingface_hub/__init__.py)

# We will use Cluster embeddings

# You can use KMeans (fixed k=5) or HDBSCAN (finds variable clusters).

In [ ]:
from sklearn.cluster import KMeans

num_clusters = 5
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
cluster_ids = kmeans.fit_predict(embeddings)

# Add cluster labels back to dataset
dataset = dataset.add_column("cluster", cluster_ids)

# Interpret clusters

In [ ]:
import pandas as pd

df = pd.DataFrame({"title": corpus, "cluster": cluster_ids})

# Show top few terms per cluster
for c in range(num_clusters):
    print(f"\nCluster {c}")
    print(df[df.cluster == c].sample(10)["title"].tolist())

Semi-supervised labeling

Manually map each cluster → one of your 5 target categories.

In [ ]:
cluster_to_category = {
    0: "Diseases & Disorders",
    1: "Anatomy & Physiology",
    2: "Pharmacology & Drugs",
    3: "Procedures & Interventions",
    4: "Mechanisms & Complications",
}

df["category"] = df["cluster"].map(cluster_to_category)